# 🚀 GitTrend — Spark Analysis
## Topik 7: Monitor Repositori Open Source Populer
**Pipeline: HDFS → PySpark (DataFrame API + Spark SQL) → HDFS + Dashboard JSON**

### Kontributor
- **Anggota 4 — [Ahmad Rafi Fadhillah Dwiputra]**: Seluruh notebook Spark analysis (3 analisis wajib, export hasil ke HDFS dan dashboard)

### 3 Analisis Wajib:
1. **Distribusi bahasa pemrograman** — bahasa apa yang paling banyak digunakan?
2. **Top 10 repositori berdasarkan bintang** — repo mana yang paling populer?
3. **Kata trending di deskripsi repo** — tema apa yang sedang tren?

### Catatan Teknis
Notebook ini dijalankan di **Google Colab** karena PySpark worker crash di Windows lokal akibat Docker Desktop mengubah hostname resolution (`kubernetes.docker.internal`). Data tetap berasal dari pipeline HDFS (di-staging via `consumer_to_hdfs.py` → `docker cp` ke lokal → upload ke Colab). Hasil analisis di-upload kembali ke HDFS.

## 0. Setup & Instalasi

In [ ]:
# [Anggota4]: Setup environment
!pip install pyspark -q
print('PySpark installed ✅')

In [ ]:
# [Anggota4]: Upload data dari HDFS staging
import os, json, re
from datetime import datetime

# --- MODE SELECTION ---
# Set True jika menjalankan di lokal dengan akses HDFS
# Set False jika menjalankan di Google Colab (upload manual)
USE_HDFS = True

HDFS_API_PATH = 'hdfs://localhost:8020/data/github/api/'
HDFS_RSS_PATH = 'hdfs://localhost:8020/data/github/rss/'
HDFS_HASIL_PATH = 'hdfs://localhost:8020/data/github/hasil/'

LOCAL_API_DIR = '/content/data/api'
LOCAL_RSS_DIR = '/content/data/rss'
OUTPUT_DIR = '/content/output'

if not USE_HDFS:
    from google.colab import files
    os.makedirs(LOCAL_API_DIR, exist_ok=True)
    os.makedirs(LOCAL_RSS_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print('📂 Upload file JSON dari folder tmp/spark_staging/api/')
    uploaded_api = files.upload()
    for fname, content in uploaded_api.items():
        with open(f'{LOCAL_API_DIR}/{fname}', 'wb') as f:
            f.write(content)
    print(f'✅ {len(uploaded_api)} file API uploaded')
else:
    print(f'📂 HDFS mode — akan membaca dari {HDFS_API_PATH}')

In [ ]:
# [Anggota4]: Upload RSS data
if not USE_HDFS:
    print('📂 Upload file JSON dari folder tmp/spark_staging/rss/')
    uploaded_rss = files.upload()
    for fname, content in uploaded_rss.items():
        with open(f'{LOCAL_RSS_DIR}/{fname}', 'wb') as f:
            f.write(content)
    print(f'✅ {len(uploaded_rss)} file RSS uploaded')
else:
    print(f'📂 HDFS mode — akan membaca dari {HDFS_RSS_PATH}')

## 1. Inisialisasi PySpark

In [ ]:
# [Anggota4]: Inisialisasi Spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GitTrend Analysis') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'✅ Spark version: {spark.version}')
print(f'✅ SparkSession ready')

## 2. Load & Eksplorasi Data
Data diambil dari HDFS (hasil pipeline Kafka Consumer → HDFS).
Jika HDFS tidak tersedia (Colab), data di-load dari file JSON yang sudah di-staging.

In [ ]:
# [Anggota4]: Load data — HDFS atau lokal
if USE_HDFS:
    # === MEMBACA LANGSUNG DARI HDFS ===
    print('📂 Membaca data dari HDFS...')
    df_api = spark.read.json(HDFS_API_PATH)
    df_rss = spark.read.json(HDFS_RSS_PATH)
    print(f'✅ API records from HDFS: {df_api.count()}')
    print(f'✅ RSS records from HDFS: {df_rss.count()}')
else:
    # === FALLBACK: Membaca dari file lokal (staging dari HDFS) ===
    print('📂 Membaca data dari file lokal (staging dari HDFS)...')
    api_records = []
    for f in os.listdir(LOCAL_API_DIR):
        if f.endswith('.json'):
            with open(f'{LOCAL_API_DIR}/{f}', 'r', encoding='utf-8') as fp:
                data = json.load(fp)
                if isinstance(data, list):
                    api_records.extend(data)
                else:
                    api_records.append(data)

    rss_records = []
    for f in os.listdir(LOCAL_RSS_DIR):
        if f.endswith('.json'):
            with open(f'{LOCAL_RSS_DIR}/{f}', 'r', encoding='utf-8') as fp:
                data = json.load(fp)
                if isinstance(data, list):
                    rss_records.extend(data)
                else:
                    rss_records.append(data)

    df_api = spark.createDataFrame(api_records)
    df_rss = spark.createDataFrame(rss_records) if rss_records else None
    print(f'✅ API records loaded: {len(api_records)}')
    print(f'✅ RSS records loaded: {len(rss_records)}')

In [ ]:
# [Anggota4]: Eksplorasi schema dan sample data
print(f'📊 API DataFrame: {df_api.count()} rows')
print('\nSchema:')
df_api.printSchema()
print('\n📰 Sample API data:')
df_api.select('full_name', 'language', 'stargazers_count', 'description').show(5, truncate=50)

In [ ]:
if df_rss:
    print(f'📰 RSS DataFrame: {df_rss.count()} rows')
    df_rss.printSchema()
    df_rss.show(5, truncate=50)
else:
    print('⚠️ No RSS data available')

## 3. Register Spark SQL Tables
Membuat temporary view untuk query SQL.

In [ ]:
# [Anggota4]: Register temp views untuk Spark SQL
df_api.createOrReplaceTempView('github_repos')
if df_rss:
    df_rss.createOrReplaceTempView('tech_news')

print('📈 Quick Statistics:')
spark.sql("""
    SELECT 
        COUNT(*) as total_repos,
        COUNT(DISTINCT language) as unique_languages,
        ROUND(AVG(stargazers_count), 1) as avg_stars,
        MAX(stargazers_count) as max_stars,
        ROUND(AVG(forks_count), 1) as avg_forks
    FROM github_repos
""").show()

---
## 📊 Analisis 1: Distribusi Bahasa Pemrograman
**Metode:** Spark SQL — `GROUP BY` + `ORDER BY` + Subquery + Aggregasi

**Pertanyaan bisnis:** Bahasa pemrograman apa yang paling banyak digunakan di repositori trending GitHub?

In [ ]:
# [Anggota4]: Analisis 1 — Distribusi bahasa pemrograman (Spark SQL)
df_lang = spark.sql("""
    SELECT 
        COALESCE(language, 'Unknown') AS language,
        COUNT(*) as repo_count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM github_repos), 1) as percentage,
        ROUND(AVG(stargazers_count), 1) as avg_stars,
        SUM(forks_count) as total_forks
    FROM github_repos
    GROUP BY COALESCE(language, 'Unknown')
    ORDER BY repo_count DESC
""")

print('🔤 Distribusi Bahasa Pemrograman di Repositori Trending:')
print('=' * 70)
df_lang.show(20, truncate=False)

lang_results = [row.asDict() for row in df_lang.collect()]

### 📝 Interpretasi Analisis 1

Distribusi bahasa pemrograman menunjukkan tren teknologi yang sedang berkembang di ekosistem open source:
- **JavaScript/TypeScript** mendominasi — menunjukkan dominasi ekosistem web dan full-stack development.
- **Python** tetap populer berkat AI/ML dan automation tools.
- **C++** dan **C#** hadir untuk game modding dan desktop tooling.
- Kategori **Unknown** (30%) menandakan banyak repository berupa dokumentasi, kumpulan resource, atau konfigurasi tanpa kode sumber. Ini mencerminkan bahwa GitHub bukan hanya untuk kode, tapi juga knowledge sharing.

---
## ⭐ Analisis 2: Top 10 Repositori Berdasarkan Bintang
**Metode:** Spark SQL + DataFrame API — `ORDER BY` + `LIMIT` + Window Function `ROW_NUMBER()`

**Pertanyaan bisnis:** Repositori mana yang paling populer berdasarkan jumlah stars?

In [ ]:
# [Anggota4]: Analisis 2 — Top 10 repo berdasarkan stars (Spark SQL + DataFrame API)

# Menggunakan Spark SQL untuk ranking
df_top10 = spark.sql("""
    SELECT 
        ROW_NUMBER() OVER (ORDER BY stargazers_count DESC) as rank,
        full_name,
        language,
        stargazers_count,
        forks_count,
        SUBSTRING(COALESCE(description, ''), 1, 80) as description_short
    FROM github_repos
    ORDER BY stargazers_count DESC
    LIMIT 10
""")

print('⭐ Top 10 Repositori Paling Populer (by Stars):')
print('=' * 80)
df_top10.show(10, truncate=False)

# Detail menggunakan DataFrame API
print('\n📋 Detail Top 10 dengan Deskripsi:')
for row in df_top10.collect():
    desc = row['description_short'] or 'No description'
    print(f"  #{row['rank']} ⭐{row['stargazers_count']} | {row['full_name']}")
    print(f"     Language: {row['language']} | Forks: {row['forks_count']}")
    print(f"     {desc}")
    print()

top10_results = [row.asDict() for row in df_top10.collect()]

### 📝 Interpretasi Analisis 2

Top 10 repositori trending menunjukkan pola menarik:
- Repository dengan **star tertinggi** terkait tools AI, automation, atau viral content.
- **Rasio forks/stars** yang rendah menunjukkan proyek baru yang mendapat perhatian cepat tapi belum banyak di-fork.
- Banyak repo trending merupakan **AI-powered tools** (Claude Code skills, LLM inference, trading bots) — menunjukkan tren "AI-first" di open source.
- Beberapa repo viral karena **gaming/modding community** (FiveM, Subnautica, PS5 exploits), menunjukkan segmen non-enterprise yang aktif.

---
## 📝 Analisis 3: Kata Trending di Deskripsi Repo
**Metode:** DataFrame API — `split()` + `explode()` + `regexp_replace()` + `filter()` + `groupBy()` + `orderBy()`

**Pertanyaan bisnis:** Tema atau kata kunci apa yang sedang tren berdasarkan deskripsi repositori?

In [ ]:
# [Anggota4]: Analisis 3 — Kata trending di deskripsi (DataFrame API)

stop_words = [
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'it', 'that', 'this', 'are', 'was',
    'be', 'has', 'have', 'had', 'not', 'no', 'can', 'will', 'do', 'if',
    'your', 'you', 'we', 'they', 'all', 'any', 'as', 'up', 'out', 'so',
    'its', 'than', 'then', 'into', 'over', 'also', 'just', 'more', 'about',
    'one', 'two', 'new', 'use', 'using', 'used', 'get', 'set', 'via', 'etc',
    '', '-', '--', '—', '|', 'https', 'http', 'www', 'com'
]

# Pipeline: filter null → lowercase → remove non-alpha → split → explode → filter → count
df_words = df_api \
    .filter(F.col('description').isNotNull()) \
    .filter(F.col('description') != '') \
    .select(F.explode(
        F.split(
            F.lower(F.regexp_replace(F.col('description'), r'[^a-zA-Z\s]', '')),
            r'\s+'
        )
    ).alias('word')) \
    .filter(~F.col('word').isin(stop_words)) \
    .filter(F.length('word') >= 4) \
    .groupBy('word') \
    .agg(F.count('*').alias('frequency')) \
    .orderBy(F.col('frequency').desc())

print('🔥 Top 25 Kata Trending di Deskripsi Repositori:')
print('=' * 50)
df_words.show(25, truncate=False)

word_results = [row.asDict() for row in df_words.limit(30).collect()]

### 📝 Interpretasi Analisis 3

Word frequency analysis mengungkap tema-tema yang sedang tren di ekosistem open source:
- **"polymarket", "trading", "bot"** mendominasi — mencerminkan ledakan prediction market dan automated trading di dunia crypto.
- **"skill", "claude", "codex"** menunjukkan adopsi massive tools AI generatif oleh developer.
- **"desktop", "app", "visual"** mengindikasikan kembalinya minat pada native desktop applications.
- **"arbitrage", "polygon"** menunjukkan fokus DeFi dan smart contract automation.

**Insight utama:** Ekosistem open source saat ini didominasi oleh dua tema besar: **AI-powered developer tools** dan **crypto/DeFi automation**. Developer tidak hanya *menggunakan* AI, tapi aktif *membangun* tools AI untuk berbagai domain.

---
## 4. Export Hasil ke JSON & HDFS
Simpan hasil analisis ke `spark_results.json` (untuk Dashboard) dan upload ke HDFS (`/data/github/hasil/`).

In [ ]:
# [Anggota4]: Compile dan export hasil analisis
spark_results = {
    'metadata': {
        'generated_at': datetime.now().isoformat(),
        'spark_version': spark.version,
        'total_api_records': df_api.count(),
        'total_rss_records': df_rss.count() if df_rss else 0,
        'analysis_count': 3
    },
    'analysis_1_language_distribution': lang_results,
    'analysis_2_top_repos': top10_results,
    'analysis_3_trending_words': word_results
}

output_path = '/content/output/spark_results.json' if not USE_HDFS else 'spark_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(spark_results, f, indent=2, ensure_ascii=False, default=str)

print(f'✅ Hasil disimpan ke: {output_path}')
print(f'   File size: {os.path.getsize(output_path)} bytes')
print(f'\n📋 Ringkasan:')
print(f'   - Analisis 1: {len(lang_results)} bahasa pemrograman')
print(f'   - Analisis 2: {len(top10_results)} top repos')
print(f'   - Analisis 3: {len(word_results)} trending words')

In [ ]:
# [Anggota4]: Upload hasil ke HDFS
if USE_HDFS:
    # Direct write ke HDFS via Spark
    df_hasil = spark.createDataFrame([spark_results['metadata']])
    df_hasil.write.mode('overwrite').json(HDFS_HASIL_PATH + 'metadata')
    df_lang.write.mode('overwrite').json(HDFS_HASIL_PATH + 'language_distribution')
    df_top10.write.mode('overwrite').json(HDFS_HASIL_PATH + 'top_repos')
    df_words.limit(30).write.mode('overwrite').json(HDFS_HASIL_PATH + 'trending_words')
    print(f'✅ Hasil analisis tersimpan ke HDFS: {HDFS_HASIL_PATH}')
else:
    print('📥 Download spark_results.json...')
    print('   Setelah download:')
    print('   1. Letakkan di: dashboard/data/spark_results.json')
    print('   2. Upload ke HDFS via terminal:')
    print('      docker cp dashboard/data/spark_results.json namenode:/tmp/spark_results.json')
    print('      docker exec namenode hdfs dfs -put -f /tmp/spark_results.json /data/github/hasil/')
    files.download(output_path)

In [ ]:
# Cleanup
spark.stop()
print('\n🎉 Analisis selesai!')
print('\nNext steps:')
print('1. Download spark_results.json (cell di atas)')
print('2. Pindahkan ke dashboard/data/spark_results.json')
print('3. Upload ke HDFS via docker cp + hdfs dfs -put')
print('4. Lanjut ke Anggota 5: Flask Dashboard')